In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy

In [2]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import euclidean_distances

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [10]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [13]:
TRAIN_FILE = "/content/drive/MyDrive/train.txt"

In [16]:
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

print("Total rows:", len(lines))
print("Sample rows:")
for x in lines[:10]:
    print(x)

Total rows: 11698
Sample rows:
METHOD_GET PATH_/api/Challenges/ PARAM_name TEXT
METHOD_GET PATH_/rest/user/whoami PARAM_fields TEXT
METHOD_GET PATH_/rest/languages
METHOD_GET PATH_/rest/user/whoami
METHOD_GET PATH_/api/Quantitys/
METHOD_GET PATH_/rest/products/search PARAM_q EMPTY
METHOD_GET PATH_/api/Challenges/ PARAM_name TEXT
METHOD_GET PATH_/rest/user/whoami PARAM_fields TEXT
METHOD_GET PATH_/rest/user/whoami
METHOD_POST PATH_/rest/user/login BODY_email EMAIL BODY_password TEXT


In [17]:
import random
random.shuffle(lines)
print(len(lines))

11698


In [18]:
train_lines, val_lines = train_test_split(lines, test_size=0.1, random_state=42)

print("Train:", len(train_lines))
print("Val:", len(val_lines))

Train: 10528
Val: 1170


In [19]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
class RequestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=64):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        return item

In [21]:
train_dataset = RequestDataset(train_lines, tokenizer, max_length=64)
val_dataset = RequestDataset(val_lines, tokenizer, max_length=64)

In [22]:
mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

DistilBertForMaskedLM(
  (activation): GELUActivation()
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0

In [23]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

In [26]:
training_args = TrainingArguments(
    output_dir="./distilbert_waf_mlm",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

In [27]:
trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

In [28]:
trainer.train()

Step,Training Loss
50,1.133609
100,0.130611
150,0.078971
200,0.052596
250,0.037500
300,0.036755
350,0.034351
400,0.028435
450,0.018668
500,0.026638


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=987, training_loss=0.08918658605462512, metrics={'train_runtime': 118.3421, 'train_samples_per_second': 266.887, 'train_steps_per_second': 8.34, 'total_flos': 523351444267008.0, 'train_loss': 0.08918658605462512, 'epoch': 3.0})

In [29]:
trainer.save_model("./distilbert_waf_mlm_final")
tokenizer.save_pretrained("./distilbert_waf_mlm_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilbert_waf_mlm_final/tokenizer_config.json',
 './distilbert_waf_mlm_final/tokenizer.json')

In [30]:
encoder = AutoModel.from_pretrained("./distilbert_waf_mlm_final")
encoder.to(device)
encoder.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: ./distilbert_waf_mlm_final
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [31]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


@torch.no_grad()
def encode_texts(texts, model, tokenizer, batch_size=64, max_length=64):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )

        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)

        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)

In [32]:
train_embeddings = encode_texts(train_lines, encoder, tokenizer)
val_embeddings = encode_texts(val_lines, encoder, tokenizer)

print(train_embeddings.shape, val_embeddings.shape)

(10528, 768) (1170, 768)


In [44]:
def extract_route_key(serialized_request: str) -> str:
    """
    Extract PATH_* token as the route key.
    Example:
      METHOD_GET PATH_/rest/products/search PARAM_q TEXT
      -> PATH_/rest/products/search
    """
    for token in serialized_request.split():
        if token.startswith("PATH_"):
            return token
    return "UNKNOWN_ROUTE"

In [45]:
from collections import Counter

train_route_keys = [extract_route_key(x) for x in train_lines]
val_route_keys = [extract_route_key(x) for x in val_lines]

route_counts = Counter(train_route_keys)
route_counts.most_common(20)

[('PATH_/rest/products/search', 2541),
 ('PATH_/rest/user/whoami', 1954),
 ('PATH_/api/Quantitys/', 1773),
 ('PATH_/rest/languages', 1523),
 ('PATH_/api/Challenges/', 1392),
 ('PATH_/api/Products/NUM', 894),
 ('PATH_/rest/products/NUM/reviews', 423),
 ('PATH_/rest/basket/NUM', 6),
 ('PATH_/rest/user/login', 4),
 ('PATH_/api/Cards/', 4),
 ('PATH_/profile', 4),
 ('PATH_/rest/user/security-question', 2),
 ('PATH_/api/Cards', 1),
 ('PATH_/api/SecurityAnswers/', 1),
 ('PATH_/profile/image/url', 1),
 ('PATH_/api/BasketItems/', 1),
 ('PATH_/api/SecurityQuestions/', 1),
 ('PATH_/api/Users/', 1),
 ('PATH_/api/Deliverys', 1),
 ('PATH_/api/Deliverys/NUM', 1)]

In [53]:
global_centroid = train_embeddings.mean(axis=0)
global_train_scores = np.linalg.norm(train_embeddings - global_centroid, axis=1)
global_threshold = np.percentile(global_train_scores, 95)

global_threshold = min(global_threshold, 3.1)

print("Global threshold:", global_threshold)

Global threshold: 3.1


In [47]:
from collections import defaultdict

route_to_indices = defaultdict(list)

for idx, route in enumerate(train_route_keys):
    route_to_indices[route].append(idx)

route_centroids = {}
route_thresholds = {}
route_stats = []

MIN_ROUTE_SAMPLES = 20   # if a route has fewer than this, use global fallback

for route, indices in route_to_indices.items():
    if len(indices) < MIN_ROUTE_SAMPLES:
        continue

    route_embeds = train_embeddings[indices]
    centroid = route_embeds.mean(axis=0)
    scores = np.linalg.norm(route_embeds - centroid, axis=1)
    threshold = np.percentile(scores, 95)

    route_centroids[route] = centroid
    route_thresholds[route] = threshold
    route_stats.append({
        "route": route,
        "count": len(indices),
        "threshold": float(threshold),
        "mean_score": float(scores.mean()),
        "p95_score": float(np.percentile(scores, 95)),
        "max_score": float(scores.max()),
    })

route_stats_df = pd.DataFrame(route_stats).sort_values("count", ascending=False)
route_stats_df.head(20)

,route,count,threshold,mean_score,p95_score,max_score
3,PATH_/rest/products/search,2541,0.616294,0.132424,0.616294,0.616294
1,PATH_/rest/user/whoami,1954,1.608575,1.061957,1.608575,1.608575
0,PATH_/api/Quantitys/,1773,0.000127,0.000127,0.000127,0.000127
6,PATH_/rest/languages,1523,0.000101,0.000101,0.000101,0.000101
2,PATH_/api/Challenges/,1392,0.000099,0.000099,0.000099,0.000099
5,PATH_/api/Products/NUM,894,0.000070,0.000070,0.000070,0.000071
4,PATH_/rest/products/NUM/reviews,423,0.000032,0.000031,0.000032,0.000032


In [48]:
def score_one_request(serialized_request: str, embedding: np.ndarray):
    route = extract_route_key(serialized_request)

    if route in route_centroids:
        centroid = route_centroids[route]
        threshold = route_thresholds[route]
        mode = "route"
    else:
        centroid = global_centroid
        threshold = global_threshold
        mode = "global_fallback"

    score = np.linalg.norm(embedding - centroid)
    verdict = "SUSPICIOUS" if score > threshold else "BENIGN"

    return {
        "route": route,
        "mode": mode,
        "score": float(score),
        "threshold": float(threshold),
        "verdict": verdict,
    }

In [49]:
def score_requests(texts, model, tokenizer):
    embeds = encode_texts(texts, model, tokenizer)
    results = []

    for text, emb in zip(texts, embeds):
        result = score_one_request(text, emb)
        result["text"] = text
        results.append(result)

    return pd.DataFrame(results)

In [50]:
val_route_results = score_requests(val_lines[:100], encoder, tokenizer)
val_route_results.head(20)

,route,mode,score,threshold,verdict,text
0,PATH_/api/Challenges/,route,0.000099,0.000099,BENIGN,METHOD_GET PATH_/api/Challenges/ PARAM_name TEXT
1,PATH_/rest/languages,route,0.000101,0.000101,BENIGN,METHOD_GET PATH_/rest/languages
2,PATH_/api/Products/NUM,route,0.000070,0.000070,BENIGN,METHOD_GET PATH_/api/Products/NUM PARAM_d TEXT
3,PATH_/rest/languages,route,0.000101,0.000101,BENIGN,METHOD_GET PATH_/rest/languages
4,PATH_/api/Quantitys/,route,0.000127,0.000127,BENIGN,METHOD_GET PATH_/api/Quantitys/
5,PATH_/rest/products/search,route,0.074180,0.616294,BENIGN,METHOD_GET PATH_/rest/products/search PARAM_q ...
6,PATH_/rest/languages,route,0.000101,0.000101,BENIGN,METHOD_GET PATH_/rest/languages
7,PATH_/api/Products/NUM,route,0.000070,0.000070,BENIGN,METHOD_GET PATH_/api/Products/NUM PARAM_d TEXT
8,PATH_/rest/products/search,route,0.074180,0.616294,BENIGN,METHOD_GET PATH_/rest/products/search PARAM_q ...
9,PATH_/rest/products/search,route,0.074180,0.616294,BENIGN,METHOD_GET PATH_/rest/products/search PARAM_q ...


In [51]:
val_route_results.sort_values("score", ascending=False).head(20)

,route,mode,score,threshold,verdict,text
22,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
30,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
40,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
38,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
49,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
42,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
69,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
80,PATH_/rest/user/whoami,route,1.608575,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami
55,PATH_/rest/user/whoami,route,0.792615,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami PARAM_fields...
39,PATH_/rest/user/whoami,route,0.792615,1.608575,BENIGN,METHOD_GET PATH_/rest/user/whoami PARAM_fields...


In [58]:
test_requests = [
    "METHOD_GET PATH_/rest/products/search PARAM_q TEXT",
    "METHOD_GET PATH_/rest/products/search PARAM_q EMPTY",
    "METHOD_POST PATH_/rest/user/login BODY_email EMAIL BODY_password TEXT",
    "METHOD_POST PATH_/api/BasketItems/ BODY_ProductId NUM BODY_BasketId NUM BODY_quantity NUM",
    "METHOD_GET PATH_/rest/products/search PARAM_q TEXT SQLI_TOKEN",
    "METHOD_POST PATH_/rest/user/login BODY_email EMAIL BODY_password TEXT BODY_extra TEXT",
    "METHOD_POST PATH_/api/BasketItems/ BODY_ProductId NUM BODY_BasketId NUM BODY_quantity NUM BODY_injected TEXT",
]

test_results = score_requests(test_requests, encoder, tokenizer)
test_results

,route,mode,score,threshold,verdict,text
0,PATH_/rest/products/search,route,0.074180,0.616294,BENIGN,METHOD_GET PATH_/rest/products/search PARAM_q ...
1,PATH_/rest/products/search,route,0.616294,0.616294,BENIGN,METHOD_GET PATH_/rest/products/search PARAM_q ...
2,PATH_/rest/user/login,group,3.320068,1.608455,SUSPICIOUS,METHOD_POST PATH_/rest/user/login BODY_email E...
3,PATH_/api/BasketItems/,global,4.247562,3.100000,SUSPICIOUS,METHOD_POST PATH_/api/BasketItems/ BODY_Produc...
4,PATH_/rest/products/search,route,1.346652,0.616294,SUSPICIOUS,METHOD_GET PATH_/rest/products/search PARAM_q ...
5,PATH_/rest/user/login,group,3.517128,1.608455,SUSPICIOUS,METHOD_POST PATH_/rest/user/login BODY_email E...
6,PATH_/api/BasketItems/,global,4.267249,3.100000,SUSPICIOUS,METHOD_POST PATH_/api/BasketItems/ BODY_Produc...


In [61]:
def map_route_group(route):
    if route.startswith("PATH_/rest/user") or route.startswith("PATH_/api/Users"):
        return "USER"
    
    elif route.startswith("PATH_/rest/products") or route.startswith("PATH_/api/Products"):
        return "PRODUCT"
    
    elif route.startswith("PATH_/rest/basket") or route.startswith("PATH_/api/BasketItems"):
        return "BASKET"
    
    elif route.startswith("PATH_/api/Quantitys") or route.startswith("PATH_/rest/languages"):
        return "META"
    
    else:
        return "OTHER"

In [62]:
group_to_indices = defaultdict(list)

for idx, route in enumerate(train_route_keys):
    group = map_route_group(route)
    group_to_indices[group].append(idx)

group_centroids = {}
group_thresholds = {}

MIN_GROUP_SAMPLES = 50

for group, indices in group_to_indices.items():
    if len(indices) < MIN_GROUP_SAMPLES:
        continue

    embeds = train_embeddings[indices]
    centroid = embeds.mean(axis=0)
    scores = np.linalg.norm(embeds - centroid, axis=1)
    threshold = np.percentile(scores, 95)

    group_centroids[group] = centroid
    group_thresholds[group] = threshold

In [63]:
def score_one_request(serialized_request, embedding):
    route = extract_route_key(serialized_request)

    # 1. route-level
    if route in route_centroids:
        centroid = route_centroids[route]
        threshold = route_thresholds[route]
        mode = "route"

    # 2. group-level
    else:
        group = map_route_group(route)
        if group in group_centroids:
            centroid = group_centroids[group]
            threshold = group_thresholds[group]
            mode = "group"

        # 3. global fallback
        else:
            centroid = global_centroid
            threshold = global_threshold
            mode = "global"

    score = np.linalg.norm(embedding - centroid)
    verdict = "SUSPICIOUS" if score >= threshold else "BENIGN"

    return {
        "route": route,
        "mode": mode,
        "score": float(score),
        "threshold": float(threshold),
        "verdict": verdict,
    }

In [74]:
for i in group_thresholds:
    print(i, group_thresholds[i])

META 2.2173183
USER 1.6084552
OTHER 0.034711972
PRODUCT 4.198968


In [64]:
attack_like_requests = [
    # benign
    "METHOD_GET PATH_/rest/products/search PARAM_q EMPTY",
    "METHOD_GET PATH_/rest/products/search PARAM_q TEXT",
    "METHOD_POST PATH_/rest/user/login BODY_email EMAIL BODY_password TEXT",

    # SQLi-like
    "METHOD_GET PATH_/rest/products/search PARAM_q TEXT RAWTOKEN_or RAWTOKEN_1_eq_1",
    "METHOD_POST PATH_/rest/user/login BODY_email TEXT BODY_password TEXT RAWTOKEN_or RAWTOKEN_1_eq_1",
    "METHOD_GET PATH_/rest/user/security-question PARAM_email TEXT RAWTOKEN_or RAWTOKEN_1_eq_1",

    # XSS-like
    "METHOD_GET PATH_/rest/products/search PARAM_q TEXT RAWTOKEN_script RAWTOKEN_alert",
    "METHOD_POST PATH_/rest/user/login BODY_email TEXT BODY_password TEXT RAWTOKEN_svg RAWTOKEN_onload",
]

In [65]:
for req in attack_like_requests:
    result = score_one_request(req, encode_texts([req], encoder, tokenizer)[0])
    print(f"{result['verdict']} | {result['mode']} | {result['score']:.2f} | {req}")

BENIGN | route | 0.62 | METHOD_GET PATH_/rest/products/search PARAM_q EMPTY
BENIGN | route | 0.07 | METHOD_GET PATH_/rest/products/search PARAM_q TEXT
SUSPICIOUS | group | 3.32 | METHOD_POST PATH_/rest/user/login BODY_email EMAIL BODY_password TEXT
SUSPICIOUS | route | 2.48 | METHOD_GET PATH_/rest/products/search PARAM_q TEXT RAWTOKEN_or RAWTOKEN_1_eq_1
SUSPICIOUS | group | 3.73 | METHOD_POST PATH_/rest/user/login BODY_email TEXT BODY_password TEXT RAWTOKEN_or RAWTOKEN_1_eq_1
SUSPICIOUS | group | 3.37 | METHOD_GET PATH_/rest/user/security-question PARAM_email TEXT RAWTOKEN_or RAWTOKEN_1_eq_1
SUSPICIOUS | route | 2.27 | METHOD_GET PATH_/rest/products/search PARAM_q TEXT RAWTOKEN_script RAWTOKEN_alert
SUSPICIOUS | group | 3.71 | METHOD_POST PATH_/rest/user/login BODY_email TEXT BODY_password TEXT RAWTOKEN_svg RAWTOKEN_onload


In [69]:
import pickle
import json
import numpy as np

# route artifacts
with open("waf_route_centroids.pkl", "wb") as f:
    pickle.dump(route_centroids, f)

with open("waf_route_thresholds.json", "w") as f:
    json.dump({k: float(v) for k, v in route_thresholds.items()}, f)

# group artifacts
with open("waf_group_centroids.pkl", "wb") as f:
    pickle.dump(group_centroids, f)

with open("waf_group_thresholds.json", "w") as f:
    json.dump({k: float(v) for k, v in group_thresholds.items()}, f)

# global artifacts
np.save("waf_global_centroid.npy", global_centroid)

with open("waf_global_threshold.json", "w") as f:
    json.dump({"threshold": float(global_threshold)}, f)

print("Saved route, group, and global artifacts to /content")

Saved route, group, and global artifacts to /content


In [70]:
import shutil
import os

SAVE_DIR = "/content/drive/MyDrive/model_artifacts2"
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_copy = [
    "waf_route_centroids.pkl",
    "waf_route_thresholds.json",
    "waf_group_centroids.pkl",
    "waf_group_thresholds.json",
    "waf_global_centroid.npy",
    "waf_global_threshold.json",
]

for file in files_to_copy:
    shutil.copy(file, SAVE_DIR)

print("Copied artifacts to:", SAVE_DIR)

Copied artifacts to: /content/drive/MyDrive/model_artifacts2


In [71]:
!ls /content/drive/MyDrive/model_artifacts2

waf_global_centroid.npy    waf_group_centroids.pkl    waf_route_centroids.pkl
waf_global_threshold.json  waf_group_thresholds.json  waf_route_thresholds.json


In [67]:
import shutil
import os

SAVE_DIR = "/content/drive/MyDrive/model_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy("waf_route_centroids.pkl", SAVE_DIR)
shutil.copy("waf_route_thresholds.json", SAVE_DIR)
shutil.copy("waf_global_centroid.npy", SAVE_DIR)
shutil.copy("waf_global_threshold.json", SAVE_DIR)

print("Copied artifacts to Drive")

Copied artifacts to Drive


In [68]:
!ls /content/drive/MyDrive/model_artifacts

waf_global_centroid.npy    waf_route_centroids.pkl
waf_global_threshold.json  waf_route_thresholds.json


In [ ]:
import os
import shutil

SAVE_DIR = "/content/drive/MyDrive/waf-project/model_artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy model folder
shutil.copytree("distilbert_waf_mlm_final", f"{SAVE_DIR}/distilbert_waf_mlm_final", dirs_exist_ok=True)

# Copy numpy + json + pickle files
shutil.copy("waf_global_centroid.npy", SAVE_DIR)
shutil.copy("waf_route_centroids.pkl", SAVE_DIR)
shutil.copy("waf_route_thresholds.json", SAVE_DIR)
shutil.copy("waf_global_threshold.json", SAVE_DIR)

print("Saved all artifacts to:", SAVE_DIR)

In [38]:
np.save("waf_centroid.npy", centroid)

with open("waf_threshold.json", "w") as f:
    json.dump({"threshold": float(threshold)}, f)

print("Saved centroid + threshold")

Saved centroid + threshold


In [42]:
DRIVE_DIR = "/content/drive/MyDrive/colab_outputs2"
!mkdir -p "$DRIVE_DIR"

In [43]:
!cp -r distilbert_waf_mlm_final "$DRIVE_DIR"/
!cp waf_centroid.npy "$DRIVE_DIR"/
!cp waf_threshold.json "$DRIVE_DIR"/
!cp train_scores.csv "$DRIVE_DIR"/
!cp val_scores.csv "$DRIVE_DIR"/